# Week 7 · Topic 4 — Parsing Real Text

**Box we are building: INPUT, finished**

Everything this week combines into one pipeline: strip, split, clean each
piece, build a dict. This is exactly what this week's DEMO does.

> **Already know this?** Skip to the *Topic 4 Challenge*.

---
## 1. The pipeline, one step at a time

In [2]:
raw_line = "  Srv-01 , used=87 , total=100  "

line = raw_line.strip()
hostname_part, rest = line.split(",", 1)
hostname = hostname_part.strip().lower()
print(hostname, "|", rest)

srv-01 |  used=87 , total=100


```
"  Srv-01 , used=87 , total=100  "
        | .strip()
"Srv-01 , used=87 , total=100"
        | .split(",", 1)
["Srv-01 ", "used=87 , total=100"]
        | .strip().lower() on the first piece, .split(",") on the rest
{"hostname": "srv-01", "used": 87.0, "total": 100.0}
```

### Your turn
Trace the same steps by hand on `"srv-03,used=45,total=100"` before running anything.

In [ ]:
data = "srv-03,used=45,total=100"
hostpart, usedtotal = data.split(",",1)
#host_name = hostpart.strip().lower()                   same output without this line
print(host_name, "|", usedtotal)


srv-03 | used=45,total=100


---
## 2. The whole pipeline as one function

In [12]:
def parse_line(raw_line):
    """Turn one messy raw status line into a clean dict."""
    line = raw_line.strip()
    hostname_part, rest = line.split(",", 1)
    hostname = hostname_part.strip().lower()

    fields = {}
    for piece in rest.split(","):
        key, value = piece.split("=")
        fields[key.strip().lower()] = float(value.strip())

    return {"hostname": hostname, "used": fields["used"], "total": fields["total"]}

print(parse_line("  Srv-01 , used=87 , total=100  "))

{'hostname': 'srv-01', 'used': 87.0, 'total': 100.0}


### Your turn
Adapt `parse_line` for a line shaped `"hostname:region:used:total"` instead (colon-separated, no key=value pairs).

In [46]:
def parse_line1(rawline):
    line = rawline.strip()
    host_part, rest_part = line.split(":",1)
    host_normalize = host_part.strip().lower()

    list = {}
    for i in rest_part.split(":"):
       print(i)
       list[i.strip().lower()] = i
       list[i] = i
       list[i] = i
    
    return {"hostname": host_normalize ,"region": list[region],"used":list[used] ,"total":list[total] }

parse_line1("sRv-01:iNdia:90:100")

   



iNdia
90
100


NameError: name 'region' is not defined

---
## 3. When parsing breaks

Three different failures, three different meanings — knowing which is which is
most of the job:

- an extra `=` in a value → `ValueError: too many values to unpack`
- a missing field → `KeyError`
- a value that is not really a number → `ValueError: could not convert string to float`

In [24]:
#parse_line("srv-01,note=server=east,used=87,total=100")
# ValueError: too many values to unpack (expected 2, got 3)

#parse_line("srv-01,used=87")
# KeyError: 'total'

#parse_line("srv-01,used=unknown,total=100")
# ValueError: could not convert string to float: 'unknown'

### Your turn
Uncomment each broken call one at a time, run it, and check the error matches what is described above.

In [ ]:
#all are described perfectly

---
## 4. Parsing every line in a file

Real messy text does not arrive as one line typed into the code — it arrives as a file of many lines. Combine `parse_line` with Week 4's `for line in f:`, and write the status check once, as its own function, rather than repeating the `if`/`elif` inside the loop.

In [25]:
def status_of(used, total, warning_at=90):
    """Return OVER LIMIT, WARNING or OK for a used/total pair."""
    percent = (used / total) * 100
    if percent >= 100:
        return "OVER LIMIT"
    elif percent >= warning_at:
        return "WARNING"
    else:
        return "OK"

with open("messy_lines.txt") as f:
    for raw_line in f:
        record = parse_line(raw_line)
        print(record["hostname"], status_of(record["used"], record["total"]))

FileNotFoundError: [Errno 2] No such file or directory: 'messy_lines.txt'

### Your turn
Save a few messy `hostname , used=.. , total=..` lines of your own to a file. Read it line by line, parse each one with `parse_line`, and print its status — without writing the cleaning logic or the status logic more than once.

---
## Topic 4 Challenge

Write `parse_line` for your own project's lane (see the project brief), and
test it against at least 3 raw lines with different spacing and case.

In [ ]:
# your code here

---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | parsing a raw survey row into a clean dict of answers |
| **Cyber Security** | parsing a raw log line into a clean dict of `{ip, attempts, result}` |
| **IT** | parsing a raw status line into a clean dict of `{hostname, used, total}` |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| assuming every real line is well-formed | the pipeline works until one day it doesn't | know which error means what, before it happens live |
| cleaning after converting instead of before | `AttributeError` — numbers have no `.strip()` | always clean the text, then convert |
| writing the pipeline as one long line | hard to debug when it breaks | keep it as named steps, like section 1, until you are confident |
| repeating the OVER LIMIT / WARNING / OK check inline for every line | the same logic typed twice, easy to make it disagree with itself | write it once as `status_of(...)` and call it |

## Go deeper

`examples/01_the_full_pipeline.py` — the five steps, then as one function
`examples/02_when_parsing_breaks.py` — the three failures, and what each means
`examples/03_parsing_a_file.py` — parse_line + status_of, applied to every line in a file
Docs: [docs.python.org — String Methods](https://docs.python.org/3/library/stdtypes.html#string-methods)

---
## Recap

| You can now | Using |
|---|---|
| clean a raw line | `.strip()`, `.lower()` |
| split it into fields | `.split(",", 1)`, `.split(",")` |
| build a clean dict from it | `parse_line(raw_line)` |
| tell parsing failures apart | `ValueError` (unpack, float) vs `KeyError` (missing field) |
| parse a whole file of messy lines | `for raw_line in f:` + `parse_line(raw_line)` |
| classify each one without repeating the logic | a `status_of(...)` function, called once per line |

**You now have all of Week 7's INPUT.** This is what starts your project in
the lab — nothing about it resets.